# CaQR 162-Circuit Campaign — NEW Materializer-Trained Ridge

This notebook uses the new CaQR/DQCC-trained, fixed TRAIN-scaler Ridge checkpoints. It runs two independent objectives and never mixes earlier routing reports with the new experiment. The two reserved final-test circuits remain untouched.

First execute the Qiskit 0.45 test suite and inspect `NEW_RIDGE_QUICKSTART_FA.md`. Exact feature parity with the friend’s training pipeline has not yet been independently verified.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
import pandas as pd
import qiskit
from IPython.display import display

REPAIR_ROOT = Path(r'C:\Users\ASUS\Documents\Quantum computing\qubit-reuse-repair')
CAQR_ROOT = Path(r'C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main')
OUT_DIR = REPAIR_ROOT / 'caqr_162_results_materializer_ridge_v2'
SCRIPT = REPAIR_ROOT / 'scripts' / 'caqr_campaign.py'

assert SCRIPT.is_file(), f'Extract the add-on ZIP inside REPAIR_ROOT: {SCRIPT}'
assert (REPAIR_ROOT / 'pilot_models' / 'swap_count_one_step_TRAIN_ONLY.json').is_file()
assert (CAQR_ROOT / 'main.py').is_file()
assert qiskit.__version__.startswith('0.45.'), qiskit.__version__
print('Python:',sys.executable,'Qiskit:',qiskit.__version__)
print('CaQR:',CAQR_ROOT,'Repair:',REPAIR_ROOT)

In [ ]:
def run_campaign(action, *options):
    env=dict(os.environ)
    env['PYTHONPATH']=str(REPAIR_ROOT / 'src') + os.pathsep + env.get('PYTHONPATH','')
    cmd=[sys.executable,str(SCRIPT),action,'--out-dir',str(OUT_DIR),*map(str,options)]
    print('RUN:', ' '.join(cmd),flush=True)
    done=subprocess.run(cmd,cwd=str(REPAIR_ROOT),env=env)
    if done.returncode!=0:
        raise RuntimeError(f'Campaign {action} failed, return code {done.returncode}')

run_campaign('roster','--caqr-root',CAQR_ROOT)
roster_df=pd.read_csv(OUT_DIR/'roster_162.csv')
print('Total:',len(roster_df))
display(roster_df.groupby('previous_split').size().rename('circuits').to_frame())
assert len(roster_df)==162

## Reuse existing CaQR screening records

The previous screening results depend on CaQR, not the fitted Ridge model. The following step copies **screening only**, never old routing outputs. The new materializer patch and new model require fresh routing in a separate results directory.

In [ ]:
from pathlib import Path
import shutil

old_results = REPAIR_ROOT / "caqr_162_results"
if old_results.exists() and (old_results / "screen" / "cases").is_dir():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    roster_src = old_results / "frozen_roster.json"
    roster_dest = OUT_DIR / "frozen_roster.json"
    if roster_dest.exists() and roster_dest.read_bytes() != roster_src.read_bytes():
        raise ValueError("Frozen roster differs; refusing to mix experiments")
    if not roster_dest.exists():
        shutil.copy2(roster_src, roster_dest)
    if not (OUT_DIR / "screen").exists():
        shutil.copytree(old_results / "screen", OUT_DIR / "screen")
    print("Existing screening copied or already available; old routing NOT copied")
else:
    print("No earlier screening found; run the Screening step below")

## Optional: Screen circuits

Skip this section if you imported the previous screening cache. Otherwise run once with `SCREEN_BATCHES = 1` and `SCREEN_BATCH_SIZE = 0` to screen every non-reserved circuit.

In [ ]:
SCREEN_BATCH_SIZE = 0  # 0 = all remaining circuits
SCREEN_BATCHES = 0     # set to 1 only if screening cache was not copied
for batch in range(SCREEN_BATCHES):
    print(f"\nScreening batch {batch + 1}/{SCREEN_BATCHES}")
    run_campaign('screen','--caqr-root',CAQR_ROOT,'--batch-size',SCREEN_BATCH_SIZE,
                 '--timeout',300,'--attempts',150)

In [ ]:
screen=pd.read_csv(OUT_DIR/'screening_162.csv')
display(screen.groupby('status',dropna=False).size().rename('n_circuits').to_frame())
print('Ready for Ridge/Routing:', int((screen['status']=='ready').sum()))
print('Not screened:',int((screen['status']=='not_screened').sum()))
view=['circuit','previous_split','family_heuristic','active_qubits','active_width',
      'reuse_edges','valid_one_step_neighbors','status','error']
display(screen[view].head(30))

## Evaluate new Ridge repair and routing

The new model consumes **raw plan features** and fixed TRAIN-only medians/means/standard deviations, not the old per-circuit normalization. Use a distinct output directory and the `materializer` model set. Each job contains an original/repair matched routing pair plus an Original Best-of-2 control.

In [ ]:
EVALUATION_BATCH_SIZE = 0   # 0 = all remaining jobs
EVALUATION_BATCHES = 1
for batch in range(EVALUATION_BATCHES):
    print(f"\nRouting batch {batch + 1}/{EVALUATION_BATCHES}")
    run_campaign('evaluate','--model-set','materializer',
                 '--batch-size',EVALUATION_BATCH_SIZE,
                 '--seeds','200,211,223','--topology','heavy_hex7',
                 '--budget',120,'--max-depth',5,'--routing-timeout',420)

In [ ]:
run_campaign('summary')
summary=json.loads((OUT_DIR/'summary_162.json').read_text(encoding='utf-8'))
print('Screening status:',summary['screen_counts'])
print('Routing job status:',summary['routing_job_counts'])
display(pd.DataFrame(summary['groups']))

## Phase 3 - Per-circuit results, including unsuccessful cases

The table below contains exactly **324 rows**: 162 circuits x 2 separate optimization objectives. `mean_guard_gain_pct` is available only when at least one successful routing seed yields a defined percentage. The `status` and `n_seeds_succeeded` fields indicate whether the planned evaluation is complete.

For any primary published aggregate, include only cases with all prescribed seeds completed, and report failures and coverage separately. Do not silently remove failed circuits from the screening denominator.


In [ ]:
results=pd.read_csv(OUT_DIR/'per_circuit_objective.csv')
assert len(results)==324, len(results)
cols=['circuit','previous_split','family_heuristic','objective','screen_status',
      'n_seeds_succeeded','n_seeds_requested','mean_guard_gain_pct',
      'mean_proposed_gain_abs','mean_guard_minus_best2_abs','status']
display(results[cols].sort_values(['objective','circuit']))
print('Results CSV:',OUT_DIR/'per_circuit_objective.csv')

In [ ]:
# Strict completed-case analysis: do not mix observed partial seeds or reserved test
complete=results[(results['n_seeds_succeeded']==results['n_seeds_requested']) &
                 (results['n_percent_defined']==results['n_seeds_requested']) &
                 (results['previous_split']!='final_test_reserved')]
print('Completed circuit-objective pairs:',len(complete))
if len(complete):
    display(complete.groupby(['objective','previous_split']).agg(
        n=('circuit','count'),
        mean_guard_gain_pct=('mean_guard_gain_pct','mean'),
        median_guard_gain_pct=('mean_guard_gain_pct','median'),
        mean_benefit_vs_best2=('mean_guard_minus_best2_abs','mean')).round(3))
else:
    print('No fully evaluated cases yet; continue the evaluation batches.')

### Reporting and reproducibility rules

- The new Ridge model's coefficients and scaler are frozen; model selection uses no routed labels during search.
- Results use a patched StaticPlanMaterializer; rerun routing in a new results directory, and never pool old and new results.
- Some circuits belong to the training dataset or related families; do not treat them as independent held-out evidence.
- The `unassigned` label is **not** proof of independence from training.
- Per-circuit mean gains from a guarded selection must be distinguished from unguarded candidate gains and from the matched Original Best-of-2 control.
- Independent feature-extraction parity with the trainer's feature code and exact post-patch compiler outputs remains an outstanding verification step.
- Record all skipped, no-reuse, no-neighbor, and failed cases.